<a href="https://colab.research.google.com/github/SujalGodse/Advance-Analytics-using-Statistics/blob/main/Advance_Analytics_using_Statistics_Day11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
from numpy import random
from collections import Counter
from numpy.linalg import inv
from numpy.linalg import eig
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import pylab
from pylab import legend
from pylab import plot, show, title, xlabel, ylabel
import scipy
from scipy import stats
from scipy.stats import binom,poisson,norm,t,expon,f
from sklearn.model_selection import train_test_split

import statsmodels
from statsmodels import stats
from statsmodels.stats import weightstats as ssw

# **3. Paired t test**

In [ ]:
before = np.array([12,14,10,9,8,13])
after = np.array([11.8,9.7,11.3,8.2,7.2,12.3])

Q. reduciom will come down by at least 2min

H0 avg(before - after) >= 2

Ha avg(before - after) < 2
(left tail test)


In [ ]:
print((before-after).mean())
# OR
d = before - after
np.mean(d)

0.9166666666666665


np.float64(0.9166666666666665)

In [ ]:
ssw.ttost_paired(before,after,low=2,upp=2)
# in output last 3 lines show left tail pvalue and above that 3 lies show right
# last 3 lines are :
# 1 line is z or t test value/statistical value --> np.float64(-1.4413160909218492)
# 2 line is p-value --> np.float64(0.10453031697930062)
# 3 line is degrees of freedom --> np.float64(5.0)

(np.float64(0.8954696830206994),
 (np.float64(-1.4413160909218492),
  np.float64(0.8954696830206994),
  np.float64(5.0)),
 (np.float64(-1.4413160909218492),
  np.float64(0.10453031697930062),
  np.float64(5.0)))

In [ ]:
scipy.stats.ttest_1samp(before-after,popmean=2,alternative='less')

TtestResult(statistic=np.float64(-1.4413160909218492), pvalue=np.float64(0.10453031697930062), df=np.int64(5))

In [ ]:
(np.mean(d)-2)/(np.std(d)/len(d)**0.5)

np.float64(-1.5788826709861274)

# **4. One way ANOVA (ANalysis Of VAriance)**

In [36]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp

In [ ]:
dfA = pd.DataFrame({'Col1':np.repeat('A',7),'Col2':np.random.randint(10,30,7)})
dfB = pd.DataFrame({'Col1':np.repeat('B',9),'Col2':np.random.randint(20,40,9)})
dfC = pd.DataFrame({'Col1':np.repeat('C',8),'Col2':np.random.randint(15,30,8)})
df = pd.concat([dfA,dfB,dfC])
df
print(df.groupby('Col1').mean())

           Col2
Col1           
A     21.714286
B     27.111111
C     21.375000


# **IMPORTANT**

In [ ]:
modl = ols('Col2 ~ Col1',df).fit()    # model creation --> col 1 is factor (Categorical data) and col2 is response (Continuous data)
# ols - ordinary least square
# value of col2 depend on value of col1 or not
tbl = sm.stats.anova_lm(modl)
print(tbl)
# last PR value is p-value
# In below output F is our test statistics

# sum_sq = col1 -> variation between sample , residual -> variation within samples
# df(degree of freedom) = col1 -> (len(sample) -1)=2 , residual -> (len(each_sample)-1) - col1_df
# mean_sq = col1 -> sum_sq_col1 / df , residual -> sum_sq_col1 / df
# F( test statistics) = mean_sq_col1 / last_mean_sq

            df     sum_sq    mean_sq         F    PR(>F)
Col1       2.0  175.43254  87.716270  3.264917  0.058259
Residual  21.0  564.19246  26.866308       NaN       NaN


F = variation between samples / variation within sample

In [ ]:
print(statsmodels.stats.multicomp.pairwise_tukeyhsd(df.Col2,df.Col1))

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     A      B   5.3968 0.1214  -1.1872 11.9809  False
     A      C  -0.3393 0.9912   -7.101  6.4224  False
     B      C  -5.7361 0.0813 -12.0845  0.6122  False
-----------------------------------------------------


Anova on excel

In [16]:
# due to indexing problem
df = pd.read_excel("/content/drive/MyDrive/CDAC/data_anova.xlsx")
df.index = np.arange(df.shape[0])

In [18]:
grp1 = df.groupby('Col1')

In [21]:
sm = grp1.agg('mean')
gm = np.mean(df.Col2)

In [22]:
# variation between sample
7*(sm.loc['A']-gm)**2 + 9*(sm.loc['B']-gm)**2 + 8*(sm.loc['C']-gm)**2

,0
Col2,842.849206


In [23]:
sampA = df.Col2[np.where(df.Col1=='A')[0]]
sampB = df.Col2[np.where(df.Col1=='B')[0]]
sampC = df.Col2[np.where(df.Col1=='C')[0]]

In [ ]:
# variation within sample
sum((np.array(sampA) - np.array(sm.loc['A']))*2) + sum((np.array(sampB) - np.array(sm.loc['B']))2) + sum((np.array(sampC) - np.array(sm.loc['C']))*2)

In [24]:
842.849 + 684.484

1527.333

In [26]:
# total = betwenn + within
sum((np.array(df.Col2)-np.array(gm))**2)

np.float64(1527.3333333333333)

# **Multiple ANOVA**

In [34]:
zoo = [34,43,57,40,85,68,67,53,41,24,42,52]
supp=[1,1,1,1,2,2,2,2,3,3,3,3]
lake=["R","D","R","D","R","D","R","D","R","D","R","D"]
df=pd.DataFrame({'Resp':zoo,"Factor1":supp,"Factor2":lake})
df

,Resp,Factor1,Factor2
0,34,1,R
1,43,1,D
2,57,1,R
3,40,1,D
4,85,2,R
5,68,2,D
6,67,2,R
7,53,2,D
8,41,3,R
9,24,3,D


In [41]:
print(df.nunique())
model = ols('Resp ~ Factor1+Factor2',df).fit()
tbl = sm.stats.anova_lm(model)
tbl

Resp       12
Factor1     3
Factor2     2
dtype: int64


,df,sum_sq,mean_sq,F,PR(>F)
Factor2,1.0,176.333333,176.333333,0.543765,0.479659
Factor1,1.0,28.125000,28.125000,0.086730,0.775056
Residual,9.0,2918.541667,324.282407,NaN,NaN


In [44]:
0.543765-0.086730

0.4570350000000001